# Project Pipeline — S&P 500 Short-Term Volatility Forecasting

This cumulative notebook connects Stage 04 through Stage 16 for the final project. It downloads raw data, cleans and stores it, builds leakage-safe features, trains and evaluates models, generates stakeholder outputs, and demonstrates the productized Flask API.

In [1]:
# --- run me first ---
from pathlib import Path
import os, sys
if Path.cwd().name == 'notebooks':
    os.chdir('..')
ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('working from:', ROOT.name)

working from: project


## Stage 01-03: Framing, setup, and reusable utilities

The README and docs define the stakeholder, decision, assumptions, lifecycle mapping, reproducible structure, and reusable utility/configuration helpers.

In [2]:
from src.config import masked_config_status
masked_config_status()

{'project_root': 'C:\\Users\\yifunger\\Documents\\Codex\\2026-08-18\\new-chat\\work\\bootcamp_Yifang_Qiu\\project',
 'data_dir': 'C:\\Users\\yifunger\\Documents\\Codex\\2026-08-18\\new-chat\\work\\bootcamp_Yifang_Qiu\\project\\data',
 'raw_data_dir': 'C:\\Users\\yifunger\\Documents\\Codex\\2026-08-18\\new-chat\\work\\bootcamp_Yifang_Qiu\\project\\data\\raw',
 'processed_data_dir': 'C:\\Users\\yifunger\\Documents\\Codex\\2026-08-18\\new-chat\\work\\bootcamp_Yifang_Qiu\\project\\data\\processed',
 'reports_dir': 'C:\\Users\\yifunger\\Documents\\Codex\\2026-08-18\\new-chat\\work\\bootcamp_Yifang_Qiu\\project\\reports',
 'model_dir': 'C:\\Users\\yifunger\\Documents\\Codex\\2026-08-18\\new-chat\\work\\bootcamp_Yifang_Qiu\\project\\model',
 'docs_dir': 'C:\\Users\\yifunger\\Documents\\Codex\\2026-08-18\\new-chat\\work\\bootcamp_Yifang_Qiu\\project\\docs',
 'env_file_present': 'False'}

## Stage 04-12: Run the integrated data-to-report pipeline

The CLI functions are imported back into the notebook so the notebook and reusable project code stay connected.

In [3]:
from src.run_step import run_all

artifacts = run_all(start_date='2018-01-01', end_date='2026-08-26')
artifacts

2026-08-27 00:23:34,910 INFO Starting ingest step


2026-08-27 00:23:35,817 INFO Finished ingest step; saved sources=['manifest', 'sp500', 'treasury_10y', 'treasury_2y', 'vix']


2026-08-27 00:23:35,821 INFO Starting clean step


2026-08-27 00:23:35,881 INFO Finished clean step with 2173 rows


2026-08-27 00:23:35,883 INFO Starting features step


...

2026-08-27 00:23:38,149 INFO Finished model step with selected model=ridge


2026-08-27 00:23:38,151 INFO Starting evaluate step


2026-08-27 00:23:39,773 INFO Finished evaluate step


2026-08-27 00:23:39,778 INFO Pipeline complete


{'ingest': {'sp500': 'data\\raw\\sp500_yahoo.csv',
  'vix': 'data\\raw\\vix_yahoo.csv',
  'treasury_10y': 'data\\raw\\treasury_10y_fred.csv',
  'treasury_2y': 'data\\raw\\treasury_2y_fred.csv',
  'manifest': 'data\\raw\\raw_data_manifest.json'},
 'clean_rows': 2173,
 'feature_rows': 2148,
 'feature_count': 14,
 'eda': {'numeric_summary': 'reports\\tables\\eda_numeric_summary.csv',
  'correlation_matrix': 'reports\\tables\\eda_correlation_matrix.csv',
  'missingness': 'reports\\tables\\eda_missingness.csv',
  'volatility_time_series': 'reports\\figures\\eda_volatility_time_series.png',
  'vix_vs_future_vol': 'reports\\figures\\eda_vix_vs_future_vol.png',
  'correlation_heatmap': 'reports\\figures\\eda_correlation_heatmap.png'},
 'selected_model': 'ridge',
 'metrics': [{'model': 'naive_last_5d_realized_vol',
   'MAE': 0.0038548789743183425,
   'RMSE': 0.0063417627376447195,
   'R2': -0.175609827130895,
   'MAE_improvement_vs_naive': 0.0,
   'RMSE_improvement_vs_naive': 0.0},
  {'model': 

## Modeling results and naive baseline comparison

In [4]:
import pandas as pd
metrics = pd.read_csv('reports/tables/model_metrics.csv')
metrics

,model,MAE,RMSE,R2,MAE_improvement_vs_naive,RMSE_improvement_vs_naive
0,naive_last_5d_realized_vol,0.003855,0.006342,-0.175610,0.000000,0.000000
1,linear_regression,0.002775,0.004688,0.357503,0.280117,0.260729
2,ridge,0.002774,0.004688,0.357601,0.280322,0.260785
3,random_forest,0.003532,0.006555,-0.255827,0.083735,-0.033554


## Evaluation, uncertainty, and scenario checks

In [5]:
scenario = pd.read_csv('reports/tables/scenario_metrics.csv')
ci = pd.read_csv('reports/tables/bootstrap_mae_ci.csv')
display(scenario)
display(ci)

,scenario,n,selected_MAE,naive_MAE,MAE_improvement_vs_naive,selected_bias
0,All test observations,430,0.002774,0.003855,0.280322,0.000168
1,Low/normal VIX regime,366,0.002446,0.003308,0.260649,0.000322
2,High VIX regime,64,0.004652,0.006982,0.333632,-0.000713
3,High realized-volatility regime,85,0.004098,0.007612,0.461593,-0.000876


,mae,bootstrap_mean,ci_lower,ci_upper,n_bootstrap
0,0.002774,0.002773,0.002426,0.003127,1000


## Stage 13: Productized API test evidence

This cell starts the Flask app, calls `/health`, `/features`, a valid `/predict`, and an intentionally bad `/predict`, then stops the server.

In [6]:
import os
import subprocess
import time
import requests
import pandas as pd

api_port = '5067'
env = os.environ.copy()
env['API_PORT'] = api_port
server = subprocess.Popen([sys.executable, 'app.py'], cwd=str(ROOT), env=env, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
base_url = f'http://127.0.0.1:{api_port}'
try:
    ready = False
    for _ in range(40):
        try:
            r = requests.get(base_url + '/health', timeout=1)
            if r.status_code == 200:
                ready = True
                break
        except requests.RequestException:
            time.sleep(0.25)
    print('API ready:', ready)
    print('Health:', requests.get(base_url + '/health', timeout=5).json())
    features = requests.get(base_url + '/features', timeout=5).json()['feature_columns']
    latest = pd.read_csv('data/processed/model_ready_volatility.csv').tail(1).iloc[0]
    payload = {'features': {feature: float(latest[feature]) for feature in features}}
    good = requests.post(base_url + '/predict', json=payload, timeout=5)
    print('POST /predict status:', good.status_code)
    print('POST /predict JSON:', good.json())
    bad = requests.post(base_url + '/predict', json={'features': [1, 2]}, timeout=5)
    print('Bad POST status:', bad.status_code)
    print('Bad POST JSON:', bad.json())
finally:
    server.terminate()
    try:
        server.wait(timeout=5)
    except subprocess.TimeoutExpired:
        server.kill()
        server.wait(timeout=5)

API ready: True
Health: {'model': 'ridge', 'status': 'ok', 'target': 'future_5d_realized_volatility'}
POST /predict status: 200
POST /predict JSON: {'model': 'ridge', 'prediction': 0.004883287839910242}
Bad POST status: 400
Bad POST JSON: {'error': "'features' must contain exactly 14 values."}


## Stage 14-16: Monitoring, orchestration, lifecycle review, and grading checklist

In [7]:
required_outputs = [
    'docs/monitoring_plan.md',
    'docs/handoff_plan.md',
    'docs/orchestration_plan.md',
    'docs/lifecycle_framework_guide.md',
    'docs/project_summary.md',
    'docs/grading_checklist.md',
    'reports/volatility_risk_report.md',
    'model/model.pkl',
]
{path: Path(path).exists() for path in required_outputs}

{'docs/monitoring_plan.md': True,
 'docs/handoff_plan.md': True,
 'docs/orchestration_plan.md': True,
 'docs/lifecycle_framework_guide.md': True,
 'docs/project_summary.md': True,
 'docs/grading_checklist.md': True,
 'reports/volatility_risk_report.md': True,
 'model/model.pkl': True}